> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 2 · Notebook 07 — Cointegration, regimes and simulation

**Sessions:** S7 (Cointegration, regimes, filters & simulation) · [Lesson plan](../../docs/lessons/PART_02_QUANT_TOOLKIT.md)

**You will:**
1. Run an Engle–Granger cointegration test.
2. Find calm and stressed regimes with a hidden Markov model, without look-ahead.
3. Estimate drawdown risk with Monte Carlo simulation.

How these notebooks work: the loading and plotting code is written for you. Cells marked **✍️ Your turn** need 1–5 lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p2lib.py is in notebooks/part02/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p2lib as p

p.use_course_style()
pd.set_option("display.float_format", "{:,.4f}".format)
prices = p.load_prices()          # dates × 10 tickers (course data via P2_DATA, else synthetic)
rets = p.log_returns(prices)      # daily log returns
prices.tail(3)

## 1. Engle–Granger: is XOM cointegrated with XLE?

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
y, x = np.log(prices["XOM"]), np.log(prices["XLE"])
beta = np.polyfit(x, y, 1)[0]
beta = p.check("hedge ratio", beta, p.hedge_ratio(y, x))

In [ ]:
from statsmodels.tsa.stattools import coint
stat, pval, _ = coint(y, x)
resid = y - beta * x
print(f"Hedge ratio {beta:.3f};  Engle–Granger p-value {pval:.2e};  residual half-life {p.half_life(resid):.1f} days")
print(f"For comparison, SPY vs QQQ: p-value {coint(np.log(prices['SPY']), np.log(prices['QQQ']))[1]:.2f}")

## 2. Regimes with a hidden Markov model

`model.predict` uses the **whole sample** (fine for describing history, look-ahead for trading). `p.hmm_forward_filter` only uses data up to each day.

In [ ]:
from hmmlearn.hmm import GaussianHMM
xr = rets["SPY"].to_numpy().reshape(-1, 1)
hmm = GaussianHMM(n_components=2, covariance_type="full", n_iter=200, random_state=0).fit(xr)
stress_state = int(np.argmax([np.sqrt(c[0, 0]) for c in hmm.covars_]))
smoothed = hmm.predict_proba(xr)[:, stress_state]                    # uses the future
filtered = p.hmm_forward_filter(hmm, xr)[:, stress_state]            # uses only the past
probs = pd.DataFrame({"Smoothed (uses future data)": smoothed, "Filtered (real-time)": filtered}, index=rets.index)
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
probs["Smoothed (uses future data)"].plot(ax=axes[0], title="P(stressed regime): smoothed", lw=1)
probs["Filtered (real-time)"].plot(ax=axes[1], title="P(stressed regime): filtered", color=p.PALETTE[1], lw=1)
for ax in axes: ax.set_xlabel(""); ax.set_ylim(-0.05, 1.05)
plt.tight_layout(); plt.show()
print(f"Days where the two disagree on the regime: {((smoothed > 0.5) != (filtered > 0.5)).mean():.1%}")

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
r = rets["SPY"].to_numpy()
vol_calm = r[filtered <= 0.5].std(ddof=1) * np.sqrt(252)
vol_stress = r[filtered > 0.5].std(ddof=1) * np.sqrt(252)
vol_calm, vol_stress = p.check("regime volatilities", (vol_calm, vol_stress), p.regime_vols(r, filtered))
print(f"Calm: {vol_calm:.1%}   Stressed: {vol_stress:.1%}")

## 3. Monte Carlo: how likely is a 25% drawdown within 5 years?

In [ ]:
MU, SIGMA, YEARS, N = 0.10, 0.18, 5, 20_000          # ✏️ change me
paths = p.gbm_paths(100, MU, SIGMA, YEARS, 252 * YEARS, N, seed=1)
print(f"Mean terminal value {paths[:, -1].mean():.1f} vs theory {100 * np.exp(MU * YEARS):.1f}")

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
dd = (paths / np.maximum.accumulate(paths, axis=1) - 1).min(axis=1)
prob_dd = (dd < -0.25).mean()
prob_dd = p.check("P(max drawdown < −25%)", prob_dd, p.mc_drawdown_prob(paths, -0.25))

In [ ]:
# Same question with bootstrapped historical SPY returns (keeps fat tails and crashes)
rng = np.random.default_rng(2)
hist = rets["SPY"].to_numpy()
boot = 100 * np.exp(np.cumsum(rng.choice(hist, size=(5000, 252 * YEARS)), axis=1))
print(f"P(max drawdown < −25%): GBM {prob_dd:.1%}  vs  bootstrap of history {p.mc_drawdown_prob(boot, -0.25):.1%}")

## Questions
1. Why is SPY vs QQQ not cointegrated even though they are highly correlated?
2. When does the filtered regime probability react later than the smoothed one? Why does that matter for a trading rule?
3. Why do the GBM and bootstrap drawdown probabilities differ?